In [1]:
import json
import os
import re
import subprocess
from pathlib import Path
from urllib.parse import urlparse

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [2]:
class DependencyHealthChecker:
    """
    DependencyHealthChecker

    This checker evaluates whether a research software artifact declares
    dependencies in a maintainable and security-conscious way.

    It checks for:
    - dependency declaration files
    - pinned or bounded dependency versions
    - unpinned dependencies
    - risky dependency patterns
    - lock files
    - dependency update tooling
    - excessive dependency footprint
    - optional pip-audit support if available

    Formal idea:
        dependencyHealth : A → {True, False}
    """

    def __init__(
        self,
        json_file,
        download_dir="downloads",
        minimum_score=4,
        default_max_dependency_count=300,
        run_pip_audit=False
    ):
        self.json_file = json_file
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)

        self.minimum_score = minimum_score
        self.default_max_dependency_count = default_max_dependency_count
        self.run_pip_audit = run_pip_audit

        self.artifacts = self.load_metadata(json_file)
        self.results = []

    def load_metadata(self, json_file):
        with open(json_file, "r", encoding="utf-8") as file:
            data = json.load(file)

        return data.get("artifacts", {})

    def is_git_repository(self, uri):
        return isinstance(uri, str) and uri.startswith("https://github.com/")

    def repo_name_from_uri(self, uri):
        parsed = urlparse(uri)
        repo_name = parsed.path.rstrip("/").split("/")[-1]

        if repo_name.endswith(".git"):
            repo_name = repo_name[:-4]

        return repo_name or "repository"

    def clone_repository(self, artifact_id, uri):
        repo_name = self.repo_name_from_uri(uri)
        target_dir = self.download_dir / repo_name

        if target_dir.exists():
            print(f"📁 Repository already exists: {target_dir}")
            return target_dir

        print(f"⬇️ Cloning repository: {uri}")

        try:
            result = subprocess.run(
                ["git", "clone", "--depth", "1", uri, str(target_dir)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                print(f"❌ Failed to clone repository for {artifact_id}")
                print(result.stderr.strip())
                return None

            print(f"✅ Cloned to: {target_dir}")
            return target_dir

        except Exception as e:
            print(f"❌ Clone error for {artifact_id}: {e}")
            return None

    def read_text_file(self, path, max_chars=300000):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as file:
                return file.read(max_chars)
        except Exception:
            return ""

    def find_existing_files(self, repo_dir, possible_paths):
        found = []

        for relative_path in possible_paths:
            path = repo_dir / relative_path
            if path.exists():
                found.append(relative_path)

        return found

    def find_files_by_name(self, repo_dir, names):
        names_lower = {name.lower() for name in names}
        found = []

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv"
                }
            ]

            for file in files:
                if file.lower() in names_lower:
                    path = Path(root) / file
                    try:
                        found.append(str(path.relative_to(repo_dir)))
                    except Exception:
                        found.append(str(path))

        return found

    def parse_requirements_txt(self, path):
        dependencies = []

        text = self.read_text_file(path)

        for line in text.splitlines():
            raw_line = line
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if line.startswith("-r ") or line.startswith("--"):
                continue

            if "://" in line and not line.startswith("#"):
                dependencies.append({
                    "raw": raw_line,
                    "name": line,
                    "source": str(path.name),
                    "pinned": False,
                    "bounded": False,
                    "risky": True,
                    "risk_reason": "Direct URL dependency"
                })
                continue

            pinned = "==" in line
            bounded = any(op in line for op in [">=", "<=", "~=", ">", "<"])
            name = re.split(r"[<>=~!;\[]", line)[0].strip()

            dependencies.append({
                "raw": raw_line,
                "name": name,
                "source": str(path.name),
                "pinned": pinned,
                "bounded": bounded,
                "risky": False,
                "risk_reason": ""
            })

        return dependencies

    def parse_environment_yml(self, path):
        dependencies = []

        text = self.read_text_file(path)

        for line in text.splitlines():
            raw_line = line
            line = line.strip()

            if not line.startswith("- "):
                continue

            dep = line.replace("- ", "", 1).strip()

            if not dep or dep in {"pip:", "python"}:
                continue

            pinned = "=" in dep
            bounded = pinned
            name = dep.split("=")[0].strip()

            dependencies.append({
                "raw": raw_line,
                "name": name,
                "source": str(path.name),
                "pinned": pinned,
                "bounded": bounded,
                "risky": False,
                "risk_reason": ""
            })

        return dependencies

    def parse_pyproject_toml(self, path):
        dependencies = []

        text = self.read_text_file(path)

        dependency_blocks = re.findall(
            r"dependencies\s*=\s*\[(.*?)\]",
            text,
            flags=re.DOTALL
        )

        for block in dependency_blocks:
            matches = re.findall(r'"([^"]+)"|\'([^\']+)\'', block)

            for match in matches:
                dep = match[0] or match[1]
                dep = dep.strip()

                if not dep:
                    continue

                pinned = "==" in dep
                bounded = any(op in dep for op in [">=", "<=", "~=", ">", "<"])
                name = re.split(r"[<>=~!;\[]", dep)[0].strip()

                dependencies.append({
                    "raw": dep,
                    "name": name,
                    "source": str(path.name),
                    "pinned": pinned,
                    "bounded": bounded,
                    "risky": False,
                    "risk_reason": ""
                })

        return dependencies

    def parse_setup_py(self, path):
        dependencies = []

        text = self.read_text_file(path)

        install_requires_match = re.search(
            r"install_requires\s*=\s*\[(.*?)\]",
            text,
            flags=re.DOTALL
        )

        if install_requires_match:
            block = install_requires_match.group(1)
            matches = re.findall(r'"([^"]+)"|\'([^\']+)\'', block)

            for match in matches:
                dep = match[0] or match[1]
                dep = dep.strip()

                if not dep:
                    continue

                pinned = "==" in dep
                bounded = any(op in dep for op in [">=", "<=", "~=", ">", "<"])
                name = re.split(r"[<>=~!;\[]", dep)[0].strip()

                dependencies.append({
                    "raw": dep,
                    "name": name,
                    "source": str(path.name),
                    "pinned": pinned,
                    "bounded": bounded,
                    "risky": False,
                    "risk_reason": ""
                })

        return dependencies

    def parse_package_json(self, path):
        dependencies = []

        try:
            with open(path, "r", encoding="utf-8") as file:
                package_data = json.load(file)
        except Exception:
            return dependencies

        for section in ["dependencies", "devDependencies", "peerDependencies", "optionalDependencies"]:
            deps = package_data.get(section, {})

            if not isinstance(deps, dict):
                continue

            for name, version in deps.items():
                version = str(version)

                pinned = re.match(r"^\d+\.\d+\.\d+", version) is not None
                bounded = any(version.startswith(prefix) for prefix in ["^", "~", ">=", "<=", ">", "<"])
                risky = version in ["*", "latest"] or "git+" in version or "://" in version

                dependencies.append({
                    "raw": f"{name}: {version}",
                    "name": name,
                    "source": str(path.name),
                    "pinned": pinned,
                    "bounded": bounded,
                    "risky": risky,
                    "risk_reason": "Wildcard/latest/git/url dependency" if risky else ""
                })

        return dependencies

    def collect_dependencies(self, repo_dir):
        dependencies = []

        requirements_files = self.find_files_by_name(repo_dir, ["requirements.txt"])
        for relative in requirements_files:
            dependencies.extend(self.parse_requirements_txt(repo_dir / relative))

        environment_files = self.find_files_by_name(repo_dir, ["environment.yml", "environment.yaml"])
        for relative in environment_files:
            dependencies.extend(self.parse_environment_yml(repo_dir / relative))

        pyproject_files = self.find_files_by_name(repo_dir, ["pyproject.toml"])
        for relative in pyproject_files:
            dependencies.extend(self.parse_pyproject_toml(repo_dir / relative))

        setup_files = self.find_files_by_name(repo_dir, ["setup.py"])
        for relative in setup_files:
            dependencies.extend(self.parse_setup_py(repo_dir / relative))

        package_files = self.find_files_by_name(repo_dir, ["package.json"])
        for relative in package_files:
            dependencies.extend(self.parse_package_json(repo_dir / relative))

        unique = []
        seen = set()

        for dep in dependencies:
            key = (dep["name"].lower(), dep["raw"])
            if key not in seen:
                unique.append(dep)
                seen.add(key)

        return unique

    def detect_dependency_files(self, repo_dir):
        dependency_file_names = [
            "requirements.txt",
            "environment.yml",
            "environment.yaml",
            "pyproject.toml",
            "setup.py",
            "setup.cfg",
            "Pipfile",
            "Pipfile.lock",
            "poetry.lock",
            "uv.lock",
            "package.json",
            "package-lock.json",
            "yarn.lock",
            "pnpm-lock.yaml",
            "renv.lock",
            "DESCRIPTION"
        ]

        return self.find_files_by_name(repo_dir, dependency_file_names)

    def detect_lock_files(self, repo_dir):
        lock_file_names = [
            "Pipfile.lock",
            "poetry.lock",
            "uv.lock",
            "requirements.lock",
            "package-lock.json",
            "yarn.lock",
            "pnpm-lock.yaml",
            "conda-lock.yml",
            "conda-lock.yaml",
            "renv.lock"
        ]

        return self.find_files_by_name(repo_dir, lock_file_names)

    def detect_update_security_files(self, repo_dir):
        update_security_paths = [
            ".github/dependabot.yml",
            ".github/dependabot.yaml",
            ".github/workflows/dependency-review.yml",
            ".github/workflows/dependency-review.yaml",
            ".github/workflows/codeql.yml",
            ".github/workflows/codeql.yaml",
            ".github/workflows/security.yml",
            ".github/workflows/security.yaml",
            ".snyk",
            "renovate.json",
            ".renovaterc",
            ".renovaterc.json"
        ]

        return self.find_existing_files(repo_dir, update_security_paths)

    def detect_risky_patterns_in_dependency_files(self, repo_dir):
        dependency_files = self.detect_dependency_files(repo_dir)
        risky_patterns = []

        patterns = {
            "wildcard_dependency": r"==\s*\*|:\s*\"\*\"|latest",
            "direct_url_dependency": r"https?://|git\+",
            "editable_install": r"^-e\s+",
            "unpinned_pip_install": r"pip install\s+[a-zA-Z0-9_\-]+(\s|$)"
        }

        for relative in dependency_files:
            path = repo_dir / relative
            text = self.read_text_file(path)

            for pattern_name, pattern in patterns.items():
                if re.search(pattern, text, flags=re.MULTILINE | re.IGNORECASE):
                    risky_patterns.append({
                        "file": relative,
                        "pattern": pattern_name
                    })

        return risky_patterns

    def run_pip_audit_if_available(self, repo_dir):
        requirements_path = repo_dir / "requirements.txt"

        if not self.run_pip_audit:
            return {
                "executed": False,
                "available": None,
                "return_code": None,
                "stdout": "",
                "stderr": "",
                "vulnerabilities_detected": None
            }

        if not requirements_path.exists():
            return {
                "executed": False,
                "available": None,
                "return_code": None,
                "stdout": "",
                "stderr": "",
                "vulnerabilities_detected": None,
                "reason": "requirements.txt not found"
            }

        try:
            check = subprocess.run(
                ["python", "-m", "pip_audit", "--version"],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )

            if check.returncode != 0:
                return {
                    "executed": False,
                    "available": False,
                    "return_code": None,
                    "stdout": "",
                    "stderr": check.stderr,
                    "vulnerabilities_detected": None,
                    "reason": "pip-audit is not installed"
                }

            result = subprocess.run(
                ["python", "-m", "pip_audit", "-r", str(requirements_path)],
                cwd=str(repo_dir),
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            vulnerabilities_detected = result.returncode != 0

            return {
                "executed": True,
                "available": True,
                "return_code": result.returncode,
                "stdout": result.stdout[-3000:],
                "stderr": result.stderr[-3000:],
                "vulnerabilities_detected": vulnerabilities_detected
            }

        except Exception as e:
            return {
                "executed": False,
                "available": None,
                "return_code": None,
                "stdout": "",
                "stderr": str(e),
                "vulnerabilities_detected": None,
                "reason": str(e)
            }

    def evaluate_dependency_health(self, repo_dir, artifact_data):
        config = artifact_data.get("dependency_health", {})

        minimum_score = config.get("minimum_score", self.minimum_score)
        max_dependency_count = config.get("max_dependency_count", self.default_max_dependency_count)

        dependency_files = self.detect_dependency_files(repo_dir)
        lock_files = self.detect_lock_files(repo_dir)
        update_security_files = self.detect_update_security_files(repo_dir)
        dependencies = self.collect_dependencies(repo_dir)
        risky_patterns = self.detect_risky_patterns_in_dependency_files(repo_dir)
        pip_audit_result = self.run_pip_audit_if_available(repo_dir)

        dependency_count = len(dependencies)
        pinned_count = sum(1 for dep in dependencies if dep["pinned"])
        bounded_count = sum(1 for dep in dependencies if dep["bounded"])
        unpinned_count = sum(1 for dep in dependencies if not dep["pinned"] and not dep["bounded"])
        risky_dependency_count = sum(1 for dep in dependencies if dep["risky"])

        score = 0
        evidence = []
        issues = []

        if dependency_files:
            score += 1
            evidence.append(f"Dependency declaration files found: {', '.join(dependency_files[:10])}")
        else:
            issues.append("No dependency declaration files found")

        if dependency_count > 0:
            score += 1
            evidence.append(f"Declared dependencies parsed: {dependency_count}")
        else:
            issues.append("No dependencies could be parsed")

        if dependency_count <= max_dependency_count:
            score += 1
            evidence.append(f"Dependency count within threshold: {dependency_count} <= {max_dependency_count}")
        else:
            issues.append(f"Dependency count exceeds threshold: {dependency_count} > {max_dependency_count}")

        if dependency_count > 0:
            bounded_ratio = (pinned_count + bounded_count) / dependency_count
        else:
            bounded_ratio = 0

        if dependency_count == 0:
            issues.append("No version-bound dependencies detected")
        elif bounded_ratio >= 0.5:
            score += 1
            evidence.append(f"At least half of dependencies are pinned or bounded: {bounded_ratio:.2f}")
        else:
            issues.append(f"Low pinned/bounded dependency ratio: {bounded_ratio:.2f}")

        if lock_files:
            score += 1
            evidence.append(f"Lock files found: {', '.join(lock_files[:10])}")
        else:
            issues.append("No lock file found")

        if update_security_files:
            score += 1
            evidence.append(f"Dependency update/security tooling found: {', '.join(update_security_files[:10])}")
        else:
            issues.append("No Dependabot/Renovate/security workflow evidence found")

        if risky_dependency_count == 0 and not risky_patterns:
            score += 1
            evidence.append("No obvious risky dependency patterns detected")
        else:
            issues.append(f"Risky dependency patterns detected: dependencies={risky_dependency_count}, files={len(risky_patterns)}")

        if pip_audit_result["executed"]:
            if pip_audit_result["vulnerabilities_detected"]:
                issues.append("pip-audit detected dependency vulnerabilities or audit failure")
            else:
                score += 1
                evidence.append("pip-audit executed without detected vulnerabilities")
        elif self.run_pip_audit:
            issues.append(f"pip-audit not executed: {pip_audit_result.get('reason', 'unknown reason')}")

        dependency_healthy = score >= minimum_score and risky_dependency_count == 0

        return {
            "dependency_healthy": dependency_healthy,
            "score": score,
            "minimum_score": minimum_score,
            "dependency_files": dependency_files,
            "lock_files": lock_files,
            "update_security_files": update_security_files,
            "dependency_count": dependency_count,
            "pinned_count": pinned_count,
            "bounded_count": bounded_count,
            "unpinned_count": unpinned_count,
            "risky_dependency_count": risky_dependency_count,
            "risky_patterns": risky_patterns,
            "pip_audit_result": pip_audit_result,
            "evidence": evidence,
            "issues": issues
        }

    def check_artifact(self, artifact_id, artifact_data):
        title = artifact_data.get("title", "")
        uri = artifact_data.get("uri", "")

        print("\n" + "=" * 80)
        print(f"🔍 Dependency Health Check for {artifact_id}")
        print(f"📦 Title: {title}")
        print(f"🔗 URI: {uri}")

        artifact_result = {
            "artifact_id": artifact_id,
            "title": title,
            "uri": uri,
            "dependency_healthy": False,
            "status": "failed"
        }

        if not self.is_git_repository(uri):
            print("❌ Unsupported artifact type for this checker.")
            artifact_result["reason"] = "Unsupported artifact type."
            return artifact_result

        repo_dir = self.clone_repository(artifact_id, uri)

        if repo_dir is None:
            artifact_result["reason"] = "Repository could not be cloned."
            artifact_result["status"] = "not_evaluated_repository_unavailable"
            return artifact_result

        result = self.evaluate_dependency_health(repo_dir, artifact_data)
        artifact_result.update(result)

        print("\n📊 Dependency health evidence:")
        print(f" - Score: {result['score']} / required {result['minimum_score']}")
        print(f" - Dependency files: {', '.join(result['dependency_files']) if result['dependency_files'] else 'None'}")
        print(f" - Lock files: {', '.join(result['lock_files']) if result['lock_files'] else 'None'}")
        print(f" - Update/security tooling: {', '.join(result['update_security_files']) if result['update_security_files'] else 'None'}")
        print(f" - Parsed dependencies: {result['dependency_count']}")
        print(f" - Pinned dependencies: {result['pinned_count']}")
        print(f" - Bounded dependencies: {result['bounded_count']}")
        print(f" - Unpinned dependencies: {result['unpinned_count']}")
        print(f" - Risky dependencies: {result['risky_dependency_count']}")

        print("\n🔎 Evidence found:")
        if result["evidence"]:
            for item in result["evidence"]:
                print(f" - {item}")
        else:
            print(" - No dependency-health evidence found.")

        print("\n⚠️ Issues / weak evidence:")
        if result["issues"]:
            for item in result["issues"]:
                print(f" - {item}")
        else:
            print(" - No major dependency-health issues detected.")

        if result["dependency_healthy"]:
            artifact_result["status"] = "passed"
            print("\n✅ Dependency Health Result: PASSED")
        else:
            artifact_result["status"] = "failed"
            print("\n❌ Dependency Health Result: FAILED")

        return artifact_result

    def run(self):
        self.results = []

        print("🌱 Starting Dependency Health Fitness Function")
        print(f"📄 Metadata file: {self.json_file}")
        print(f"📁 Download directory: {self.download_dir}")

        for artifact_id, artifact_data in self.artifacts.items():
            result = self.check_artifact(artifact_id, artifact_data)
            self.results.append(result)

        print("\n" + "=" * 80)
        print("📌 Dependency Health Summary")
        print("=" * 80)

        for result in self.results:
            icon = "✅" if result["dependency_healthy"] else "❌"
            print(f"{icon} {result['artifact_id']}: {result['status']}")

        return self.results

In [3]:
checker = DependencyHealthChecker(
    json_file="artifacts.json",
    download_dir="downloads",
    minimum_score=4,
    default_max_dependency_count=300,
    run_pip_audit=True
)

dependency_health_results = checker.run()

🌱 Starting Dependency Health Fitness Function
📄 Metadata file: artifacts.json
📁 Download directory: downloads

🔍 Dependency Health Check for artifact_1
📦 Title: We provide our resources in a dedicated repository
🔗 URI: https://github.com/hihey54/hicss58
📁 Repository already exists: downloads/hicss58

📊 Dependency health evidence:
 - Score: 5 / required 4
 - Dependency files: requirements.txt
 - Lock files: None
 - Update/security tooling: None
 - Parsed dependencies: 3
 - Pinned dependencies: 0
 - Bounded dependencies: 3
 - Unpinned dependencies: 0
 - Risky dependencies: 0

🔎 Evidence found:
 - Dependency declaration files found: requirements.txt
 - Declared dependencies parsed: 3
 - Dependency count within threshold: 3 <= 300
 - At least half of dependencies are pinned or bounded: 1.00
 - No obvious risky dependency patterns detected

⚠️ Issues / weak evidence:
 - No lock file found
 - No Dependabot/Renovate/security workflow evidence found
 - pip-audit detected dependency vulnerabili

In [5]:
if PANDAS_AVAILABLE:
    df = pd.DataFrame(dependency_health_results)

    columns_to_show = [
        "artifact_id",
        "title",
        "dependency_healthy",
        "status",
        "score",
        "minimum_score",
        "dependency_count",
        "pinned_count",
        "bounded_count",
        "unpinned_count",
        "risky_dependency_count",
        "dependency_files",
        "lock_files",
        "update_security_files"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]
    display(df[existing_columns])
else:
    for result in dependency_health_results:
        print(result)

,artifact_id,title,dependency_healthy,status,score,minimum_score,dependency_count,pinned_count,bounded_count,unpinned_count,risky_dependency_count,dependency_files,lock_files,update_security_files
0,artifact_1,We provide our resources in a dedicated reposi...,True,passed,5.0,4.0,3.0,0.0,3.0,0.0,0.0,[requirements.txt],[],[]
1,artifact_2,Trending Customer Dataset,False,not_evaluated_repository_unavailable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,artifact_3,Python algorithms,True,passed,6.0,4.0,21.0,0.0,21.0,0.0,0.0,"[uv.lock, pyproject.toml]",[uv.lock],[.github/dependabot.yml]
3,artifact_4,Scikit-learn,True,passed,4.0,4.0,13.0,0.0,5.0,8.0,0.0,"[pyproject.toml, doc/binder/requirements.txt, ...",[],"[.github/dependabot.yml, .github/workflows/cod..."
4,artifact_5,Pandas,True,passed,5.0,4.0,94.0,50.0,51.0,41.0,0.0,"[environment.yml, pyproject.toml]",[],"[.github/dependabot.yml, .github/workflows/cod..."
5,artifact_6,NumPy,True,passed,5.0,4.0,37.0,11.0,11.0,26.0,0.0,"[environment.yml, pyproject.toml, numpy/f2py/s...",[],"[.github/dependabot.yml, .github/workflows/dep..."
6,artifact_7,Matplotlib,True,passed,5.0,4.0,73.0,26.0,39.0,34.0,0.0,"[environment.yml, pyproject.toml, lib/matplotl...",[],[.github/dependabot.yml]
7,artifact_8,Scrapy,False,failed,4.0,4.0,89.0,70.0,16.0,5.0,3.0,"[pyproject.toml, docs/requirements.txt]",[],[]
8,artifact_9,Flask,True,passed,5.0,4.0,28.0,21.0,6.0,1.0,0.0,"[uv.lock, pyproject.toml, examples/tutorial/py...",[uv.lock],[]
9,artifact_10,TensorFlow,False,failed,5.0,4.0,41.0,33.0,5.0,3.0,1.0,[third_party/xla/xla/backends/cpu/benchmarks/e...,[],[.github/dependabot.yml]


In [6]:
output_file = "dependency_health_results.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(dependency_health_results, file, indent=4)

print(f"✅ Results saved to {output_file}")

✅ Results saved to dependency_health_results.json
